# Comparative PCM Analysis: Network Model & Dispatch Mode Effects

**90-Day Prescient Simulation Comparison (Jan–Mar 2019, ERCOT 2035 Coal Retirement)**

This notebook compares three Prescient simulation scenarios on the TX-123BT system:

| Scenario | Duration | Network Model | Dispatch Mode | Description |
|----------|----------|---------------|---------------|-------------|
| **PTDF UC+ED** | 90 days | PTDF | UC + SCED | New network formulation with economic dispatch |
| **Btheta UC-only** | 90 days | B-theta | UC only | Traditional DC power flow, no SCED |
| **Benchmark** | 365 days | B-theta | UC + SCED | Full-year reference (Q1 subset used) |

**Key questions:**
1. How does the PTDF network model affect LMPs and congestion patterns vs B-theta?
2. What is the impact of removing economic dispatch (UC-only) on prices and costs?
3. Do the 90-day Q1 results align with the Q1 subset of the 365-day benchmark?

**Outputs:**
- `Bus_LMP_ptdf.csv`, `Bus_LMP_uc_only.csv` — wide-format LMP data per scenario
- `Generator_Dispatch_ptdf.csv`, `Generator_Dispatch_uc_only.csv` — dispatch data
- `PCM_result_ptdf.json`, `PCM_result_uc_only.json` — summary statistics

## 1. Imports & Paths

Configure paths to the three simulation result directories and shared metadata.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# === SCENARIO PATHS ===
_base = os.path.join(os.path.dirname(os.getcwd()),
                     "data", "retirement_allowed_no_extreme_half_load", "Prescient_2")

results_ptdf = os.path.join(_base, "results")
results_uc_only = os.path.join(_base, "results_uc_only")
results_benchmark = os.path.join(
    "/Users/yilu/Documents/development/nd/research/gtep/123_bus_coal/results",
    "retirement_allowed_no_extreme", "12mon_no_hydro_with_curtailment"
)

# Shared metadata (gen.csv, bus.csv, branch.csv from Prescient_2)
prescient_dir = _base

# Output directory
output_dir = os.getcwd()

# Per-scenario detail file paths
scenarios = {
    'PTDF UC+ED': {
        'results_path': results_ptdf,
        'bus_detail': os.path.join(results_ptdf, 'bus_detail.csv'),
        'thermal_detail': os.path.join(results_ptdf, 'thermal_detail.csv'),
        'renew_detail': os.path.join(results_ptdf, 'renewables_detail.csv'),
        'suffix': 'ptdf',
    },
    'Btheta UC-only': {
        'results_path': results_uc_only,
        'bus_detail': os.path.join(results_uc_only, 'bus_detail.csv'),
        'thermal_detail': os.path.join(results_uc_only, 'thermal_detail.csv'),
        'renew_detail': os.path.join(results_uc_only, 'renewables_detail.csv'),
        'suffix': 'uc_only',
    },
}

# Verify all paths exist
for name, s in scenarios.items():
    print(f"\n{name}:")
    for key in ['bus_detail', 'thermal_detail', 'renew_detail']:
        exists = os.path.exists(s[key])
        print(f"  {key}: {'OK' if exists else 'MISSING'}")

_bench_exists = os.path.exists(results_benchmark)
print(f"\nBenchmark: {'OK' if _bench_exists else 'MISSING — benchmark comparisons will be skipped'}")
if not _bench_exists:
    warnings.warn(f"Benchmark path not found: {results_benchmark}. "
                  "Update results_benchmark in Cell 2 for your system.")
print(f"Metadata dir: {'OK' if os.path.exists(os.path.join(prescient_dir, 'gen.csv')) else 'MISSING'}")


## 2. Helper Functions

Core extraction functions adapted from `prescient_lmp_analysis.ipynb`.

In [ ]:
def _prescient_output_to_df(file_name):
    """Load Prescient output CSV and combine Date/Hour/Minute into Datetime."""
    df = pd.read_csv(file_name)
    df['Datetime'] = (
        pd.to_datetime(df['Date'])
        + pd.to_timedelta(df['Hour'], 'hour')
        + pd.to_timedelta(df['Minute'], 'minute')
    )
    df.drop(columns=['Date', 'Hour', 'Minute'], inplace=True)
    cols = df.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    return df[cols]


def make_lmp_csv(lmp_path, bus_details_path, bus_name, output_dir="."):
    """Extract LMP for a single bus and append to the wide-format LMP CSV."""
    out_csv = os.path.join(output_dir, os.path.basename(lmp_path) if lmp_path else "Bus_LMP.csv")
    bdf = _prescient_output_to_df(bus_details_path)
    bdf = bdf[bdf["Bus"] == bus_name][["Datetime", "LMP", "LMP DA"]]
    bdf.set_index("Datetime", inplace=True)
    bdf = bdf.rename(columns={'LMP': f'{bus_name}_LMP', 'LMP DA': f'{bus_name}_LMP DA'})

    if lmp_path is None:
        bdf.to_csv(out_csv)
    else:
        lmp_df = pd.read_csv(lmp_path).set_index("Datetime")
        if f"{bus_name}_LMP" in lmp_df.columns:
            return
        bdf_aligned = bdf.reindex(lmp_df.index)
        lmp_df = pd.concat([lmp_df, bdf_aligned], axis=1)
        lmp_df.to_csv(out_csv)


def make_dispatch_csv(dispatch_path, gen_details_path, gen_name, gen_type,
                      other_info=None, output_dir="."):
    """Extract dispatch for a single generator and append to wide-format CSV."""
    out_csv = os.path.join(output_dir,
                           os.path.basename(dispatch_path) if dispatch_path else "Generator_Dispatch.csv")
    gdf = _prescient_output_to_df(gen_details_path)

    if gen_type == "fossil":
        info_list = ["Datetime", "Dispatch", "Dispatch DA"]
    elif gen_type == "renew":
        info_list = ["Datetime", "Output", "Output DA"]
    else:
        raise ValueError(f"Unknown gen_type: {gen_type}")

    if other_info is not None:
        info_list.extend(other_info)

    gdf_filtered = gdf[gdf["Generator"] == gen_name]
    if len(gdf_filtered) == 0:
        try:
            gdf_filtered = gdf[gdf["Generator"] == int(gen_name)]
        except (ValueError, TypeError):
            pass
    if len(gdf_filtered) == 0:
        print(f"WARNING: Generator {gen_name} not found")
        return

    gdf_filtered = gdf_filtered[info_list]
    gdf_filtered.set_index("Datetime", inplace=True)
    new_col_name = {col: f"{gen_name}_{col}" for col in info_list if col != "Datetime"}
    gdf_filtered = gdf_filtered.rename(columns=new_col_name)

    if dispatch_path is None:
        gdf_filtered.to_csv(out_csv)
    else:
        dispatch_df = pd.read_csv(dispatch_path).set_index("Datetime")
        first_col = list(new_col_name.values())[0]
        if first_col in dispatch_df.columns:
            return
        gdf_aligned = gdf_filtered.reindex(dispatch_df.index)
        dispatch_df = pd.concat([dispatch_df, gdf_aligned], axis=1)
        dispatch_df.to_csv(out_csv)


def load_weighted_lmp(bus_df, lmp_col='LMP DA', demand_col='Demand', low_threshold=1.0):
    """Compute load-weighted system LMP. Guard against near-zero demand."""
    total_demand = bus_df[demand_col].sum()
    if total_demand < low_threshold:
        return bus_df[lmp_col].mean()
    return (bus_df[demand_col] * bus_df[lmp_col]).sum() / total_demand

## 3. Discover Buses and Generators

Load shared metadata and discover unique buses/generators from the PTDF scenario.

In [ ]:
# Load shared metadata
gen_meta = pd.read_csv(os.path.join(prescient_dir, 'gen.csv'))
gen_meta['GEN UID'] = gen_meta['GEN UID'].astype(str)
bus_meta = pd.read_csv(os.path.join(prescient_dir, 'bus.csv'))
branch_df = pd.read_csv(os.path.join(prescient_dir, 'branch.csv'))
branch_df['UID'] = branch_df['UID'].astype(int)

bus_id_to_name = dict(zip(bus_meta['Bus ID'], bus_meta['Bus Name']))
bus_id_to_zone = dict(zip(bus_meta['Bus ID'], bus_meta['Zone']))

print(f"gen.csv: {len(gen_meta)} generators")
print(f"  Fuel types: {gen_meta['Fuel'].value_counts().to_dict()}")
print(f"bus.csv: {len(bus_meta)} buses")
print(f"  Zones: {bus_meta['Zone'].value_counts().to_dict()}")
print(f"branch.csv: {len(branch_df)} lines")

# Discover from PTDF scenario (both scenarios use same fleet)
bdf_raw = pd.read_csv(scenarios['PTDF UC+ED']['bus_detail'])
bus_names = sorted(bdf_raw["Bus"].unique().tolist())
print(f"\nFound {len(bus_names)} buses in bus_detail")

tdf_raw = pd.read_csv(scenarios['PTDF UC+ED']['thermal_detail'])
thermal_gens = sorted(tdf_raw["Generator"].unique().tolist())
print(f"Found {len(thermal_gens)} thermal generators")

rdf_raw = pd.read_csv(scenarios['PTDF UC+ED']['renew_detail'])
renew_gens = sorted(rdf_raw["Generator"].unique().tolist())
print(f"Found {len(renew_gens)} renewable generators")

# Build bus-name lookups
bus_detail_names = bdf_raw['Bus'].unique()
bus_name_to_zone = {}
bus_name_to_id = {}
for _, row in bus_meta.iterrows():
    for bdn in bus_detail_names:
        if row['Bus Name'] in bdn:
            bus_name_to_zone[bdn] = row['Zone']
            bus_name_to_id[bdn] = row['Bus ID']
            break
print(f"Mapped {len(bus_name_to_zone)}/{len(bus_detail_names)} bus names to zones")

del bdf_raw, tdf_raw, rdf_raw  # free memory

## 4. Extract LMP for All Buses

Generate `Bus_LMP_ptdf.csv` and `Bus_LMP_uc_only.csv` in wide format.

In [ ]:
for scenario_name, s in scenarios.items():
    suffix = s['suffix']
    lmp_csv_name = f"Bus_LMP_{suffix}.csv"
    lmp_csv_path = os.path.join(output_dir, lmp_csv_name)

    if os.path.exists(lmp_csv_path):
        print(f"{lmp_csv_name} already exists, skipping extraction.")
        continue

    print(f"\nExtracting LMP for {scenario_name}...")
    for idx, bus_name in enumerate(bus_names):
        if idx == 0:
            make_lmp_csv(lmp_path=None, bus_details_path=s['bus_detail'],
                         bus_name=bus_name, output_dir=output_dir)
            os.rename(os.path.join(output_dir, "Bus_LMP.csv"), lmp_csv_path)
        else:
            make_lmp_csv(lmp_path=lmp_csv_path, bus_details_path=s['bus_detail'],
                         bus_name=bus_name, output_dir=output_dir)
        if (idx + 1) % 25 == 0:
            print(f"  {idx + 1}/{len(bus_names)} buses done")

    df_check = pd.read_csv(lmp_csv_path)
    print(f"  {lmp_csv_name}: {df_check.shape[0]} rows x {df_check.shape[1]} columns")

## 5. Extract Dispatch for All Generators

Generate `Generator_Dispatch_ptdf.csv` and `Generator_Dispatch_uc_only.csv`.

In [ ]:
for scenario_name, s in scenarios.items():
    suffix = s['suffix']
    disp_csv_name = f"Generator_Dispatch_{suffix}.csv"
    disp_csv_path = os.path.join(output_dir, disp_csv_name)

    if os.path.exists(disp_csv_path):
        print(f"{disp_csv_name} already exists, skipping extraction.")
        continue

    print(f"\nExtracting dispatch for {scenario_name}...")

    # Thermal generators
    thermal_other_info = ["Unit Cost", "Unit State"]
    for idx, gen_name in enumerate(thermal_gens):
        if idx == 0:
            make_dispatch_csv(dispatch_path=None, gen_details_path=s['thermal_detail'],
                              gen_name=gen_name, gen_type="fossil",
                              other_info=thermal_other_info, output_dir=output_dir)
            os.rename(os.path.join(output_dir, "Generator_Dispatch.csv"), disp_csv_path)
        else:
            make_dispatch_csv(dispatch_path=disp_csv_path, gen_details_path=s['thermal_detail'],
                              gen_name=gen_name, gen_type="fossil",
                              other_info=thermal_other_info, output_dir=output_dir)

    # Renewable generators
    renew_other_info = ["Curtailment"]
    for gen_name in renew_gens:
        make_dispatch_csv(dispatch_path=disp_csv_path, gen_details_path=s['renew_detail'],
                          gen_name=gen_name, gen_type="renew",
                          other_info=renew_other_info, output_dir=output_dir)

    df_check = pd.read_csv(disp_csv_path)
    print(f"  {disp_csv_name}: {df_check.shape[0]} rows x {df_check.shape[1]} columns")

## 6. Summary Statistics

Compute and compare LMP and dispatch statistics across both 90-day scenarios.

In [ ]:
results_data = {}  # store per-scenario summary data

for scenario_name, s in scenarios.items():
    suffix = s['suffix']
    lmp_csv = os.path.join(output_dir, f"Bus_LMP_{suffix}.csv")
    disp_csv = os.path.join(output_dir, f"Generator_Dispatch_{suffix}.csv")
    df_lmp = pd.read_csv(lmp_csv)
    df_dispatch = pd.read_csv(disp_csv)

    # LMP summary per bus
    LMP_result = {}
    for bus_name in bus_names:
        lmp_da_col = f"{bus_name}_LMP DA"
        lmp_col = f"{bus_name}_LMP"
        if lmp_da_col not in df_lmp.columns:
            continue
        LMP_result[bus_name] = {
            "LMP_DA_mean": df_lmp[lmp_da_col].mean(),
            "LMP_DA_median": df_lmp[lmp_da_col].median(),
            "LMP_DA_min": df_lmp[lmp_da_col].min(),
            "LMP_DA_max": df_lmp[lmp_da_col].max(),
            "LMP_mean": df_lmp[lmp_col].mean(),
            "LMP_median": df_lmp[lmp_col].median(),
            "LMP_min": df_lmp[lmp_col].min(),
            "LMP_max": df_lmp[lmp_col].max(),
        }

    # Dispatch summary per generator
    dispatch_result = {}
    for gen_name in thermal_gens:
        gen_str = str(gen_name)
        da_col = f"{gen_str}_Dispatch DA"
        rt_col = f"{gen_str}_Dispatch"
        if da_col in df_dispatch.columns:
            dispatch_result[gen_str] = {
                "type": "thermal",
                "tot_Dispatch_DA": float(df_dispatch[da_col].sum()),
                "tot_Dispatch": float(df_dispatch[rt_col].sum()),
            }
    for gen_name in renew_gens:
        gen_str = str(gen_name)
        da_col = f"{gen_str}_Output DA"
        rt_col = f"{gen_str}_Output"
        if da_col in df_dispatch.columns:
            entry = {
                "type": "renewable",
                "tot_Output_DA": float(df_dispatch[da_col].sum()),
                "tot_Output": float(df_dispatch[rt_col].sum()),
            }
            curt_col = f"{gen_str}_Curtailment"
            if curt_col in df_dispatch.columns:
                entry["tot_Curtailment"] = float(df_dispatch[curt_col].sum())
            dispatch_result[gen_str] = entry

    results_data[scenario_name] = {
        'LMP_result': LMP_result,
        'dispatch_result': dispatch_result,
        'df_lmp': df_lmp,
        'df_dispatch': df_dispatch,
    }

    print(f"\n=== {scenario_name} ===")
    print(f"  LMP stats for {len(LMP_result)} buses")
    print(f"  Dispatch stats for {len(dispatch_result)} generators")

# Side-by-side comparison
print("\n" + "=" * 80)
print("SUMMARY COMPARISON")
print("=" * 80)
for scenario_name in scenarios:
    lmp_r = results_data[scenario_name]['LMP_result']
    all_means = [v['LMP_DA_mean'] for v in lmp_r.values()]
    print(f"\n{scenario_name}:")
    print(f"  Simple mean of bus-mean DA LMP: ${np.mean(all_means):.2f}/MWh")
    print(f"  Median of bus-mean DA LMP:      ${np.median(all_means):.2f}/MWh")
    print(f"  Range: ${min(all_means):.2f} to ${max(all_means):.2f}")

## 7. Save Summary JSON

In [ ]:
for scenario_name, s in scenarios.items():
    suffix = s['suffix']
    result_summary = {
        "LMP": results_data[scenario_name]['LMP_result'],
        "Dispatch": results_data[scenario_name]['dispatch_result'],
    }
    path = os.path.join(output_dir, f"PCM_result_{suffix}.json")
    with open(path, "w") as f:
        json.dump(result_summary, f, indent=2)
    print(f"Saved {path}")

## 8. Quick Sanity Check

Visual comparison of LMP time series and distributions across both scenarios.

In [ ]:
# Pick 4 representative buses (different zones)
sample_zones = ['WEST', 'NCENT', 'COAST', 'FWEST']
sample_buses = []
for zone in sample_zones:
    for bdn in bus_names:
        if bus_name_to_zone.get(bdn) == zone:
            sample_buses.append(bdn)
            break
if len(sample_buses) < 4:
    sample_buses = bus_names[:4]

fig, axes = plt.subplots(len(sample_buses), 1, figsize=(16, 3.5 * len(sample_buses)), sharex=True)
colors = {'PTDF UC+ED': 'steelblue', 'Btheta UC-only': 'darkorange'}

for ax, bus_name in zip(axes, sample_buses):
    zone = bus_name_to_zone.get(bus_name, '?')
    for scenario_name in scenarios:
        df = results_data[scenario_name]['df_lmp']
        df['Datetime'] = pd.to_datetime(df['Datetime'])
        col = f"{bus_name}_LMP DA"
        if col in df.columns:
            ax.plot(df['Datetime'], df[col], label=scenario_name,
                    alpha=0.7, linewidth=0.6, color=colors[scenario_name])
    ax.set_ylabel("$/MWh")
    ax.set_title(f"{bus_name} ({zone})")
    ax.legend(loc="upper right", fontsize=8)
    ax.axhline(0, color='black', linewidth=0.3)

axes[-1].set_xlabel("Date")
fig.suptitle("Day-Ahead LMP Time Series: PTDF vs Btheta UC-only", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
plt.close('all')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = {'PTDF UC+ED': 'steelblue', 'Btheta UC-only': 'darkorange'}

# (a) Mean DA LMP by bus — side by side
ax = axes[0]
for i, scenario_name in enumerate(scenarios):
    lmp_r = results_data[scenario_name]['LMP_result']
    means = [lmp_r[b]['LMP_DA_mean'] for b in bus_names if b in lmp_r]
    offset = -0.2 + i * 0.4
    ax.bar([x + offset for x in range(len(means))], means, width=0.35,
           label=scenario_name, color=colors[scenario_name], alpha=0.7)
ax.set_ylabel("Mean DA LMP ($/MWh)")
ax.set_title("Mean DA LMP by Bus")
ax.legend()
ax.set_xticks([])
ax.set_xlabel(f"{len(bus_names)} buses (sorted)")

# (b) Overlaid histogram of all bus-hour LMPs
ax = axes[1]
for scenario_name in scenarios:
    df = results_data[scenario_name]['df_lmp']
    all_lmps = []
    for b in bus_names:
        col = f"{b}_LMP DA"
        if col in df.columns:
            all_lmps.extend(df[col].dropna().tolist())
    ax.hist(all_lmps, bins=200, alpha=0.5, label=scenario_name,
            color=colors[scenario_name], density=True)
ax.set_yscale('log')
ax.set_xlabel("DA LMP ($/MWh)")
ax.set_ylabel("Density (log scale)")
ax.set_title("Distribution of All Bus-Hour DA LMPs")
ax.legend()
ax.axvline(0, color='red', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()
plt.close('all')


## 9. Negative LMP Investigation

This section investigates negative LMPs in both 90-day scenarios. With only Q1 data (Jan–Mar), we expect spring months with high renewable output and low demand to dominate the negative-price pattern.

**Comparison axes:**
- **PTDF UC+ED**: Full dispatch with PTDF network → more accurate congestion modeling
- **Btheta UC-only**: No economic dispatch, simpler network → may show different price patterns

**Investigation roadmap:**
1. Severity & prevalence in each scenario
2. Spatial patterns by ERCOT zone
3. Temporal patterns (hour-of-day, month)
4. Overgeneration & renewable correlation
5. Thermal inflexibility (coal/nuclear min-load)
6. Renewable curtailment status
7. Transmission congestion differences
8. Worst-bus deep dive
9. Conclusion

In [ ]:
# Load bus_detail for both 90-day scenarios
bus_dfs = {}
for scenario_name, s in scenarios.items():
    bdf = _prescient_output_to_df(s['bus_detail'])
    print(f"{scenario_name}: {bdf.shape[0]:,} rows, {bdf.shape[1]} columns")
    print(f"  Date range: {bdf['Datetime'].min()} to {bdf['Datetime'].max()}")
    bus_dfs[scenario_name] = bdf

# Load benchmark Q1 data (365-day run, filtered to Jan-Mar)
_bench_available = os.path.exists(results_benchmark)
if _bench_available:
    _bench_bus = _prescient_output_to_df(os.path.join(results_benchmark, 'bus_detail.csv'))
    bus_dfs['Benchmark Q1'] = _bench_bus[_bench_bus['Datetime'].dt.month <= 3].copy()
    del _bench_bus
    print(f"\nBenchmark Q1: {bus_dfs['Benchmark Q1'].shape[0]:,} rows")
    print(f"  Date range: {bus_dfs['Benchmark Q1']['Datetime'].min()} to {bus_dfs['Benchmark Q1']['Datetime'].max()}")
else:
    print("\nBenchmark path not found — benchmark comparisons will be skipped.")

# Master color map for all scenarios
colors_map = {
    'PTDF UC+ED': 'steelblue',
    'Btheta UC-only': 'darkorange',
}
if _bench_available:
    colors_map['Benchmark Q1'] = 'green'

# All scenario names for looping
all_scenario_names = list(scenarios.keys())
if _bench_available:
    all_scenario_names.append('Benchmark Q1')

print(f"\nScenarios for comparison: {all_scenario_names}")

### 9.1 Severity and Prevalence

Quantify negative LMP frequency and intensity in each scenario.

In [ ]:
lmp_col_name = 'LMP DA'

for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name]
    total_hours = len(bus_df)
    neg_hours = (bus_df[lmp_col_name] < 0).sum()
    deep_neg = (bus_df[lmp_col_name] < -100).sum()
    floor_hours = (bus_df[lmp_col_name] == -1000).sum()

    _sys_demand = bus_df['Demand'].sum()
    if _sys_demand > 1.0:
        _sys_wt_lmp = (bus_df['Demand'] * bus_df[lmp_col_name]).sum() / _sys_demand
    else:
        _sys_wt_lmp = bus_df[lmp_col_name].mean()

    print(f"\n=== {scenario_name} ===")
    print(f"Total bus-hours:        {total_hours:>12,}")
    print(f"LMP < 0:                {neg_hours:>12,}  ({100*neg_hours/total_hours:.1f}%)")
    print(f"LMP < -$100:            {deep_neg:>12,}  ({100*deep_neg/total_hours:.1f}%)")
    print(f"LMP = -$1000 (floor):   {floor_hours:>12,}  ({100*floor_hours/total_hours:.1f}%)")
    print(f"Load-weighted mean LMP: ${_sys_wt_lmp:.2f}/MWh")
    print(f"Simple mean LMP:        ${bus_df[lmp_col_name].mean():.2f}/MWh")
    print(f"Median LMP:             ${bus_df[lmp_col_name].median():.2f}/MWh")

# Side-by-side histograms
n_scen = len(all_scenario_names)
fig, axes = plt.subplots(1, n_scen, figsize=(6 * n_scen, 5), sharey=True)
if n_scen == 1:
    axes = [axes]
for ax, scenario_name in zip(axes, all_scenario_names):
    bus_df = bus_dfs[scenario_name]
    c = colors_map[scenario_name]
    ax.hist(bus_df[lmp_col_name], bins=200, edgecolor='none', alpha=0.7, color=c)
    ax.set_yscale('log')
    ax.axvline(0, color='red', linestyle='--', linewidth=1)
    ax.axvline(-1000, color='darkred', linestyle=':', linewidth=1.5)
    ax.set_xlabel('DA LMP ($/MWh)')
    ax.set_ylabel('Bus-hour count (log scale)')
    ax.set_title(f'{scenario_name}')
plt.suptitle('Distribution of Day-Ahead LMPs', fontsize=13)
plt.tight_layout()
plt.show()
plt.close('all')

### 9.2 Spatial Patterns: Which Zones Are Affected?

Compare zone-level LMP statistics between PTDF and Btheta UC-only scenarios.

In [ ]:
zone_colors = {
    'FWEST': '#d62728', 'NORTH': '#ff7f0e', 'WEST': '#e377c2',
    'NCENT': '#2ca02c', 'EAST': '#1f77b4', 'SCENT': '#9467bd',
    'SOUTH': '#8c564b', 'COAST': '#17becf'
}

bus_lmp_stats_all = {}
for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name]
    stats = bus_df.groupby('Bus')[lmp_col_name].agg(['mean', 'median', 'min', 'max', 'count'])
    stats['zone'] = stats.index.map(lambda b: bus_name_to_zone.get(b, '?'))
    stats['pct_negative'] = bus_df.groupby('Bus').apply(
        lambda g: (g[lmp_col_name] < 0).mean() * 100
    )
    stats['pct_floor'] = bus_df.groupby('Bus').apply(
        lambda g: (g[lmp_col_name] == -1000).mean() * 100
    )
    bus_lmp_stats_all[scenario_name] = stats

# Zone-level comparison
n_scen = len(all_scenario_names)
fig, axes = plt.subplots(1, n_scen, figsize=(6 * n_scen, 6))
if n_scen == 1:
    axes = [axes]
for ax, scenario_name in zip(axes, all_scenario_names):
    stats = bus_lmp_stats_all[scenario_name]
    zone_stats = stats.groupby('zone').agg(
        mean_lmp=('mean', 'mean'),
        avg_pct_floor=('pct_floor', 'mean'),
    ).sort_values('mean_lmp')
    colors = [zone_colors.get(z, 'gray') for z in zone_stats.index]
    ax.barh(zone_stats.index, zone_stats['mean_lmp'], color=colors)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Mean DA LMP ($/MWh)')
    ax.set_title(f'{scenario_name}\nMean DA LMP by Zone')
plt.tight_layout()
plt.show()
plt.close('all')

# Worst 10 buses per scenario
for scenario_name in all_scenario_names:
    stats = bus_lmp_stats_all[scenario_name]
    worst_10 = stats.sort_values('mean').head(10)
    print(f"\n=== {scenario_name}: 10 Worst Buses ===")
    print(worst_10[['zone', 'mean', 'median', 'pct_negative', 'pct_floor']].to_string(
        float_format=lambda x: f'{x:.1f}'
    ))

### 9.3 Temporal Patterns: When Do Negative LMPs Occur?

With only Q1 data (Jan–Mar), we examine hour-of-day and monthly patterns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (a) Load-weighted LMP by hour of day
ax = axes[0, 0]
for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name].copy()
    bus_df['Hour'] = bus_df['Datetime'].dt.hour
    bus_df['_wt_lmp'] = bus_df['Demand'] * bus_df[lmp_col_name]
    _h = bus_df.groupby('Hour').agg(_wt_sum=('_wt_lmp', 'sum'), _d_sum=('Demand', 'sum'))
    hourly_avg = (_h['_wt_sum'] / _h['_d_sum']).replace([np.inf, -np.inf], 0).fillna(0)
    ax.plot(hourly_avg.index, hourly_avg.values, 'o-', color=colors_map[scenario_name],
            linewidth=2, label=scenario_name, markersize=4)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Load-Weighted DA LMP ($/MWh)')
ax.set_title('System Load-Weighted DA LMP by Hour')
ax.set_xticks(range(0, 24))
ax.legend(fontsize=8)

# (b) Load-weighted LMP by month
ax = axes[0, 1]
bar_w = 0.8 / len(all_scenario_names)
for i, scenario_name in enumerate(all_scenario_names):
    bus_df = bus_dfs[scenario_name].copy()
    bus_df['Month'] = bus_df['Datetime'].dt.month
    bus_df['_wt_lmp'] = bus_df['Demand'] * bus_df[lmp_col_name]
    _m = bus_df.groupby('Month').agg(_wt_sum=('_wt_lmp', 'sum'), _d_sum=('Demand', 'sum'))
    monthly_avg = (_m['_wt_sum'] / _m['_d_sum']).replace([np.inf, -np.inf], 0).fillna(0)
    offset = (i - len(all_scenario_names)/2 + 0.5) * bar_w
    ax.bar(monthly_avg.index + offset, monthly_avg.values, width=bar_w,
           color=colors_map[scenario_name], label=scenario_name, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Month')
ax.set_ylabel('Load-Weighted DA LMP ($/MWh)')
ax.set_title('Monthly Load-Weighted DA LMP')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Jan', 'Feb', 'Mar'])
ax.legend(fontsize=8)

# (c) % buses negative by hour — all scenarios
ax = axes[1, 0]
for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name].copy()
    bus_df['Hour'] = bus_df['Datetime'].dt.hour
    pct_neg = bus_df.groupby('Hour').apply(lambda g: (g[lmp_col_name] < 0).mean() * 100)
    ax.plot(pct_neg.index, pct_neg.values, 'o-', color=colors_map[scenario_name],
            linewidth=2, label=scenario_name, markersize=4)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('% of bus-hours with LMP < 0')
ax.set_title('Negative LMP Frequency by Hour')
ax.set_xticks(range(0, 24))
ax.legend(fontsize=8)

# (d) Heatmap for first scenario (PTDF)
ax = axes[1, 1]
bus_df = bus_dfs['PTDF UC+ED'].copy()
bus_df['Hour'] = bus_df['Datetime'].dt.hour
bus_df['Month'] = bus_df['Datetime'].dt.month
bus_df['_wt_lmp'] = bus_df['Demand'] * bus_df[lmp_col_name]
_mh = bus_df.groupby(['Month', 'Hour']).agg(_wt_sum=('_wt_lmp', 'sum'), _d_sum=('Demand', 'sum'))
_mh_lmp = (_mh['_wt_sum'] / _mh['_d_sum']).replace([np.inf, -np.inf], 0).fillna(0)
pivot = _mh_lmp.unstack()
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
               vmin=-200, vmax=100, origin='lower')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Month')
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(['Jan', 'Feb', 'Mar'][:pivot.shape[0]])
ax.set_title('PTDF UC+ED: Load-Weighted LMP Heatmap')
plt.colorbar(im, ax=ax, label='$/MWh')

plt.tight_layout()
plt.show()
plt.close('all')

### 9.4 Overgeneration and Renewable Output Correlation

Compare how overgeneration and renewable output relate to LMP in each scenario.

In [ ]:
sys_hourlys = {}
for scenario_name in all_scenario_names:
    if scenario_name in scenarios:
        rp = scenarios[scenario_name]['results_path']
    else:
        rp = results_benchmark

    bus_df = bus_dfs[scenario_name].copy()
    hourly_summary = pd.read_csv(os.path.join(rp, 'hourly_summary.csv'))
    hourly_summary['Datetime'] = (
        pd.to_datetime(hourly_summary['Date'])
        + pd.to_timedelta(hourly_summary['Hour'], 'hour')
    )
    # Filter benchmark to Q1
    if scenario_name == 'Benchmark Q1':
        hourly_summary = hourly_summary[pd.to_datetime(hourly_summary['Date']).dt.month <= 3]

    bus_df['_wt_lmp'] = bus_df['Demand'] * bus_df[lmp_col_name]
    sys_hourly = bus_df.groupby('Datetime').agg(
        sys_demand=('Demand', 'sum'),
        sys_overgen=('Overgeneration', 'sum'),
        _wt_lmp_sum=('_wt_lmp', 'sum'),
    ).reset_index()
    sys_hourly['sys_mean_lmp'] = np.where(
        sys_hourly['sys_demand'] > 1.0,
        sys_hourly['_wt_lmp_sum'] / sys_hourly['sys_demand'],
        0
    )
    sys_hourly.drop(columns='_wt_lmp_sum', inplace=True)

    sys_hourly = sys_hourly.merge(
        hourly_summary[['Datetime', 'RenewablesUsed', 'OverGeneration']],
        on='Datetime', how='left'
    )
    sys_hourlys[scenario_name] = sys_hourly

# Scatter plots: 2 rows x N scenarios
n_scen = len(all_scenario_names)
fig, axes = plt.subplots(2, n_scen, figsize=(6 * n_scen, 10))
if n_scen == 1:
    axes = axes.reshape(2, 1)
for col_idx, scenario_name in enumerate(all_scenario_names):
    sh = sys_hourlys[scenario_name]
    c = colors_map[scenario_name]

    ax = axes[0, col_idx]
    ax.scatter(sh['sys_overgen'], sh['sys_mean_lmp'], s=5, alpha=0.4, color=c)
    ax.set_xlabel('System Overgeneration (MW)')
    ax.set_ylabel('Load-Weighted DA LMP ($/MWh)')
    r = sh[['sys_mean_lmp', 'sys_overgen']].corr().iloc[0, 1]
    ax.set_title(f'{scenario_name}\nLMP vs Overgen (r={r:+.3f})')

    ax = axes[1, col_idx]
    ax.scatter(sh['RenewablesUsed'], sh['sys_mean_lmp'], s=5, alpha=0.4, color=c)
    ax.set_xlabel('Renewables Used (MW)')
    ax.set_ylabel('Load-Weighted DA LMP ($/MWh)')
    r2 = sh[['sys_mean_lmp', 'RenewablesUsed']].dropna().corr().iloc[0, 1]
    ax.set_title(f'{scenario_name}\nLMP vs Renewables (r={r2:+.3f})')

plt.tight_layout()
plt.show()
plt.close('all')

### 9.5 Thermal Generator Inflexibility: Coal & Nuclear Min-Load

Coal and nuclear units cannot easily shut down, creating a "must-run" floor.

In [ ]:
# Load thermal dispatch for all scenarios
thermal_dfs = {}
renew_dfs = {}
for scenario_name in all_scenario_names:
    if scenario_name in scenarios:
        tp = scenarios[scenario_name]['thermal_detail']
        rp = scenarios[scenario_name]['renew_detail']
    else:
        tp = os.path.join(results_benchmark, 'thermal_detail.csv')
        rp = os.path.join(results_benchmark, 'renewables_detail.csv')

    tdf = _prescient_output_to_df(tp)
    tdf['Generator'] = tdf['Generator'].astype(str)
    if scenario_name == 'Benchmark Q1':
        tdf = tdf[tdf['Datetime'].dt.month <= 3]
    thermal_dfs[scenario_name] = tdf

    rdf = _prescient_output_to_df(rp)
    rdf['Generator'] = rdf['Generator'].astype(str)
    if scenario_name == 'Benchmark Q1':
        rdf = rdf[rdf['Datetime'].dt.month <= 3]
    renew_dfs[scenario_name] = rdf

# Coal & nuclear fleet
coal_nuc_gens = gen_meta[gen_meta['Fuel'].isin(['C', 'N'])][
    ['GEN UID', 'Bus ID', 'Fuel', 'PMax MW', 'PMin MW',
     'Min Down Time Hr', 'Min Up Time Hr']
].copy()
coal_nuc_gens['PMin_ratio'] = (coal_nuc_gens['PMin MW'] / coal_nuc_gens['PMax MW'] * 100).round(1)

print("=== Coal & Nuclear Fleet ===")
print(coal_nuc_gens.to_string(index=False))
print(f"\nTotal coal+nuc PMax: {coal_nuc_gens['PMax MW'].sum():.0f} MW")
print(f"Total coal+nuc PMin: {coal_nuc_gens['PMin MW'].sum():.0f} MW")

coal_nuc_ids = set(coal_nuc_gens['GEN UID'].astype(str))
for scenario_name in all_scenario_names:
    neg_lmp_hours = set(
        sys_hourlys[scenario_name][sys_hourlys[scenario_name]['sys_mean_lmp'] < 0]['Datetime']
    )
    thermal_cn = thermal_dfs[scenario_name][
        thermal_dfs[scenario_name]['Generator'].isin(coal_nuc_ids)
    ].copy()
    thermal_cn['is_neg_lmp'] = thermal_cn['Datetime'].isin(neg_lmp_hours)
    thermal_cn['is_committed'] = thermal_cn['Dispatch'] > 0

    neg_dispatch = thermal_cn[thermal_cn['is_neg_lmp']].groupby('Generator').agg(
        mean_dispatch=('Dispatch', 'mean'),
        pct_committed=('is_committed', 'mean'),
    ).reset_index()
    neg_dispatch['pct_committed'] *= 100

    print(f"\n=== {scenario_name}: Coal/Nuclear During Negative LMP Hours ({len(neg_lmp_hours)} hours) ===")
    print(f"  Mean committed dispatch: {neg_dispatch['mean_dispatch'].mean():.1f} MW per unit")

### 9.6 Curtailment Analysis

Both simulations report zero renewable curtailment, meaning all renewable output is must-take.

In [ ]:
print("=== Simulation-Wide Summary ===\n")
for scenario_name in all_scenario_names:
    if scenario_name in scenarios:
        rp = scenarios[scenario_name]['results_path']
    else:
        rp = results_benchmark
    sim_output = pd.read_csv(os.path.join(rp, 'overall_simulation_output.csv'))
    label = scenario_name
    if scenario_name == 'Benchmark Q1':
        label += ' (365d total, not Q1-filtered)'
    print(f"{label}:")
    print(f"  Total renewables curtailment: {sim_output['Total renewables curtailment'].iloc[0]:,.0f} MWh")
    print(f"  Total over generation:        {sim_output['Total over generation'].iloc[0]:,.1f} MWh")
    print(f"  Total demand:                 {sim_output['Total demand'].iloc[0]:,.0f} MWh")
    print(f"  Renewables penetration:       {sim_output['Overall renewables penetration rate'].iloc[0]*100:.1f}%")
    print()

### 9.7 Transmission Congestion: Lines at Capacity

**Key comparison**: PTDF network modeling identifies different congestion patterns than B-theta. PTDF accounts for power transfer distribution factors while B-theta uses simplified DC power flow.

In [ ]:
congestion_freqs = {}
for scenario_name in all_scenario_names:
    if scenario_name in scenarios:
        rp = scenarios[scenario_name]['results_path']
    else:
        rp = results_benchmark

    line_df = _prescient_output_to_df(os.path.join(rp, 'line_detail.csv'))
    line_df['Line'] = line_df['Line'].astype(int)
    if scenario_name == 'Benchmark Q1':
        line_df = line_df[line_df['Datetime'].dt.month <= 3]

    line_rated = line_df.merge(
        branch_df[['UID', 'From Bus', 'To Bus', 'Cont Rating']],
        left_on='Line', right_on='UID', how='left'
    )
    line_rated = line_rated[line_rated['Cont Rating'] > 0].copy()
    line_rated['utilization'] = line_rated['Flow'].abs() / line_rated['Cont Rating']
    line_rated['at_capacity'] = line_rated['utilization'] >= 0.95

    congestion_freq = line_rated.groupby('Line').agg(
        pct_at_capacity=('at_capacity', 'mean'),
        from_bus=('From Bus', 'first'),
        to_bus=('To Bus', 'first'),
        rating=('Cont Rating', 'first'),
    ).reset_index()
    congestion_freq['pct_at_capacity'] *= 100
    congestion_freq['from_name'] = congestion_freq['from_bus'].map(bus_id_to_name)
    congestion_freq['to_name'] = congestion_freq['to_bus'].map(bus_id_to_name)
    congestion_freq['from_zone'] = congestion_freq['from_bus'].map(bus_id_to_zone)
    congestion_freq['to_zone'] = congestion_freq['to_bus'].map(bus_id_to_zone)
    congestion_freqs[scenario_name] = congestion_freq

    n_congested = (congestion_freq['pct_at_capacity'] > 10).sum()
    top5 = congestion_freq.sort_values('pct_at_capacity', ascending=False).head(5)
    print(f"\n=== {scenario_name}: {n_congested} lines at >=95% capacity >10% of hours ===")
    print(top5[['Line', 'from_name', 'from_zone', 'to_name', 'to_zone',
                 'rating', 'pct_at_capacity']].to_string(index=False, float_format=lambda x: f'{x:.1f}'))

# Side-by-side congestion histograms
n_scen = len(all_scenario_names)
fig, axes = plt.subplots(1, n_scen, figsize=(6 * n_scen, 5), sharey=True)
if n_scen == 1:
    axes = [axes]
for ax, scenario_name in zip(axes, all_scenario_names):
    cf = congestion_freqs[scenario_name]
    c = colors_map[scenario_name]
    ax.hist(cf['pct_at_capacity'], bins=50, color=c, edgecolor='none', alpha=0.8)
    ax.set_xlabel('% of Hours at >= 95% Capacity')
    ax.set_ylabel('Number of Lines')
    ax.set_title(f'{scenario_name}')
plt.suptitle('Transmission Line Congestion Frequency', fontsize=13)
plt.tight_layout()
plt.show()
plt.close('all')

### 9.7b Geographic Map: Bus Locations, Zones, and Worst LMP Buses

Texas map showing all 123 buses colored by ERCOT zone, transmission lines, and the 5 worst-LMP buses highlighted. Congested lines (>50% of hours at capacity) shown in red.

In [ ]:
from matplotlib.collections import LineCollection

# Build bus coordinate lookup
bus_coords = {}
for _, row in bus_meta.iterrows():
    bus_coords[row['Bus ID']] = (row['Bus Long'], row['Bus Lat'])

# Build line segments from branch data
line_segments = []
line_ids = []
for _, row in branch_df.iterrows():
    fid, tid = int(row['From Bus']), int(row['To Bus'])
    if fid in bus_coords and tid in bus_coords:
        line_segments.append([bus_coords[fid], bus_coords[tid]])
        line_ids.append(int(row['UID']))

# Zone centroids for labels
zone_centroids = {}
for _, row in bus_meta.iterrows():
    z = row['Zone']
    if z not in zone_centroids:
        zone_centroids[z] = {'lons': [], 'lats': []}
    zone_centroids[z]['lons'].append(row['Bus Long'])
    zone_centroids[z]['lats'].append(row['Bus Lat'])
for z in zone_centroids:
    zone_centroids[z] = (
        np.mean(zone_centroids[z]['lons']),
        np.mean(zone_centroids[z]['lats'])
    )

n_scen = len(all_scenario_names)
fig, axes = plt.subplots(1, n_scen, figsize=(7 * n_scen, 8))
if n_scen == 1:
    axes = [axes]

for ax, scenario_name in zip(axes, all_scenario_names):
    stats = bus_lmp_stats_all[scenario_name]
    cf = congestion_freqs[scenario_name]

    # Build congestion lookup by line UID
    cong_lookup = dict(zip(cf['Line'], cf['pct_at_capacity']))

    # Draw all transmission lines — gray for normal, red for congested
    normal_segs, congested_segs, congested_widths = [], [], []
    for seg, lid in zip(line_segments, line_ids):
        pct = cong_lookup.get(lid, 0)
        if pct > 50:
            congested_segs.append(seg)
            congested_widths.append(1 + pct / 20)
        else:
            normal_segs.append(seg)

    if normal_segs:
        lc_normal = LineCollection(normal_segs, colors='lightgray', linewidths=0.5, zorder=1)
        ax.add_collection(lc_normal)
    if congested_segs:
        lc_cong = LineCollection(congested_segs, colors='red', linewidths=congested_widths,
                                  alpha=0.6, zorder=2)
        ax.add_collection(lc_cong)

    # Plot all buses colored by zone
    for _, row in bus_meta.iterrows():
        bname_matches = [bdn for bdn in stats.index if row['Bus Name'] in bdn]
        z = row['Zone']
        c = zone_colors.get(z, 'gray')
        ax.scatter(row['Bus Long'], row['Bus Lat'], c=c, s=25, zorder=3,
                   edgecolors='white', linewidths=0.3)

    # Highlight 5 worst buses
    worst_5 = stats.sort_values('mean').head(5)
    for bus_name in worst_5.index:
        bid = bus_name_to_id.get(bus_name)
        if bid and bid in bus_coords:
            lon, lat = bus_coords[bid]
            ax.scatter(lon, lat, marker='*', s=250, c='red', edgecolors='black',
                       linewidths=1, zorder=5)
            ax.annotate(bus_name.replace(' 345', ''),
                        (lon, lat), fontsize=6, fontweight='bold',
                        xytext=(5, 5), textcoords='offset points',
                        color='darkred', zorder=6)

    # Zone labels
    for z, (cx, cy) in zone_centroids.items():
        ax.text(cx, cy, z, fontsize=9, fontweight='bold', color='gray',
                ha='center', va='center', alpha=0.7, zorder=4,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.5))

    ax.set_xlim(-107, -93)
    ax.set_ylim(25.5, 37)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{scenario_name}\n(red lines = >50% hours congested, stars = 5 worst LMP buses)')
    ax.set_aspect('equal')

# Legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [Patch(facecolor=zone_colors[z], label=z) for z in sorted(zone_colors)]
legend_elements.append(Line2D([0], [0], color='red', linewidth=2, label='Congested line'))
legend_elements.append(Line2D([0], [0], marker='*', color='red', markersize=12,
                               linestyle='', label='5 worst LMP buses'))
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=7, ncol=2)

plt.suptitle('TX-123BT System: Bus Locations, Zones, and Congestion', fontsize=14)
plt.tight_layout()
plt.show()
plt.close('all')

### 9.8 Deep Dive: Worst Bus

Identify and compare the worst-LMP bus in each scenario.

In [ ]:
for scenario_name in all_scenario_names:
    stats = bus_lmp_stats_all[scenario_name]
    worst_bus = stats['mean'].idxmin()
    zone = bus_name_to_zone.get(worst_bus, '?')
    bus_id = bus_name_to_id.get(worst_bus)

    print(f"\n=== {scenario_name}: Worst Bus ===")
    print(f"  {worst_bus} (Bus {bus_id}, Zone: {zone})")
    print(f"  Mean DA LMP: ${stats.loc[worst_bus, 'mean']:.1f}/MWh")
    print(f"  % hours LMP < 0: {stats.loc[worst_bus, 'pct_negative']:.1f}%")
    print(f"  % hours at floor: {stats.loc[worst_bus, 'pct_floor']:.1f}%")

    local_gens = gen_meta[gen_meta['Bus ID'] == bus_id]
    print(f"  Connected generators: {len(local_gens)}")
    if len(local_gens) > 0:
        print(local_gens[['GEN UID', 'Fuel', 'PMax MW']].to_string(index=False))

# Worst bus deep dive — overlay all scenarios
stats_ptdf = bus_lmp_stats_all['PTDF UC+ED']
worst_bus_name = stats_ptdf['mean'].idxmin()

# 2-week window around worst period in PTDF
bus_data_ptdf = bus_dfs['PTDF UC+ED'][bus_dfs['PTDF UC+ED']['Bus'] == worst_bus_name].set_index('Datetime').sort_index()
worst_day = bus_data_ptdf[lmp_col_name].rolling(24, min_periods=1).mean().idxmin()
window_start = max(worst_day - pd.Timedelta(days=7), bus_data_ptdf.index.min())
window_end = min(worst_day + pd.Timedelta(days=7), bus_data_ptdf.index.max())

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

ax = axes[0]
for scenario_name in all_scenario_names:
    bus_data = bus_dfs[scenario_name][bus_dfs[scenario_name]['Bus'] == worst_bus_name].set_index('Datetime').sort_index()
    c = colors_map[scenario_name]
    ax.plot(bus_data.loc[window_start:window_end, lmp_col_name],
            color=c, linewidth=0.8, label=scenario_name, alpha=0.8)
ax.axhline(0, color='red', linewidth=0.5, linestyle='--')
ax.axhline(-1000, color='darkred', linewidth=0.5, linestyle=':')
ax.set_ylabel('DA LMP ($/MWh)')
ax.set_title(f'Worst Bus: {worst_bus_name} — 2-Week Window')
ax.legend(fontsize=8)

ax = axes[1]
for scenario_name in all_scenario_names:
    bus_data = bus_dfs[scenario_name][bus_dfs[scenario_name]['Bus'] == worst_bus_name].set_index('Datetime').sort_index()
    c = colors_map[scenario_name]
    ax.plot(bus_data.loc[window_start:window_end, 'Demand'],
            color=c, linewidth=0.8, label=scenario_name, alpha=0.8)
ax.set_ylabel('Local Demand (MW)')
ax.set_xlabel('Date')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
plt.close('all')

### 9.9 Comparative Analysis: Worst Bus vs Best Bus

In [ ]:
for scenario_name in all_scenario_names:
    stats = bus_lmp_stats_all[scenario_name]
    worst = stats['mean'].idxmin()
    best = stats['mean'].idxmax()

    print(f"\n=== {scenario_name} ===")
    comparison = pd.DataFrame({
        'Metric': ['Zone', 'Mean DA LMP', '% LMP < 0', '% at floor'],
        worst: [
            bus_name_to_zone.get(worst, '?'),
            f"${stats.loc[worst, 'mean']:.1f}",
            f"{stats.loc[worst, 'pct_negative']:.1f}%",
            f"{stats.loc[worst, 'pct_floor']:.1f}%",
        ],
        best: [
            bus_name_to_zone.get(best, '?'),
            f"${stats.loc[best, 'mean']:.1f}",
            f"{stats.loc[best, 'pct_negative']:.1f}%",
            f"{stats.loc[best, 'pct_floor']:.1f}%",
        ],
    })
    print(comparison.to_string(index=False))

# Rolling LMP for worst bus — all scenarios
worst_bus = bus_lmp_stats_all['PTDF UC+ED']['mean'].idxmin()
fig, ax = plt.subplots(figsize=(16, 5))
for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name]
    c = colors_map[scenario_name]
    ts = bus_df[bus_df['Bus'] == worst_bus].set_index('Datetime')[lmp_col_name].sort_index()
    ax.plot(ts.index, ts.rolling(24).mean(),
            color=c, alpha=0.7, linewidth=0.8, label=f'{scenario_name}')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('DA LMP ($/MWh, 24h rolling avg)')
ax.set_title(f'Worst Bus ({worst_bus}): 24h Rolling LMP — All Scenarios')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
plt.close('all')

### 9.10 Conclusion

The negative LMP patterns in both 90-day scenarios are consistent with the 365-day benchmark analysis:

1. **Spatial**: Negative LMPs concentrate in FWEST, NORTH, and WEST zones — renewable-heavy areas behind limited transmission
2. **Temporal**: Spring months (Feb–Mar) and midday hours dominate negative pricing
3. **Curtailment**: Zero curtailment in both scenarios forces all renewable output to be absorbed
4. **Network model effect**: PTDF may identify different congestion bottlenecks than B-theta, potentially affecting which buses experience the worst negative LMPs
5. **Dispatch mode effect**: UC-only lacks economic dispatch, so prices reflect only unit commitment shadow prices

## 10. Scenario Comparison: PTDF vs B-theta, UC+ED vs UC-only

This section provides a systematic 3-way comparison of simulation metrics, decomposing effects into:
- **Network model effect**: PTDF vs B-theta (compare PTDF UC+ED vs Benchmark Q1 btheta UC+ED)
- **Dispatch mode effect**: UC+ED vs UC-only (compare Benchmark Q1 btheta UC+ED vs Btheta UC-only)
- **Combined effect**: PTDF UC+ED vs Btheta UC-only

### 10.1 Overall Simulation Metrics

Key metrics from `overall_simulation_output.csv` across all three scenarios.

In [ ]:
# Load overall simulation outputs
sim_outputs = {}
for scenario_name, s in scenarios.items():
    sim_outputs[scenario_name] = pd.read_csv(os.path.join(s['results_path'], 'overall_simulation_output.csv'))

# Load benchmark
benchmark_sim = pd.read_csv(os.path.join(results_benchmark, 'overall_simulation_output.csv'))

# Key metrics
metrics = [
    'Total demand', 'Total fixed costs', 'Total generation costs', 'Total costs',
    'Total load shedding', 'Total over generation', 'Total reserve shortfall',
    'Total renewables curtailment', 'Total on/offs', 'Overall renewables penetration rate',
    'Cumulative average price',
]

comparison_rows = []
for m in metrics:
    row = {'Metric': m}
    for sn in scenarios:
        row[sn] = sim_outputs[sn][m].iloc[0]
    row['Benchmark (365d)'] = benchmark_sim[m].iloc[0]
    # Pro-rata estimate (uniform scaling) — actual Q1 values in Section 10.6
    _PRORATE_KW = {'cost', 'demand', 'generation', 'shedding', 'over gen',
                    'shortfall', 'curtailment', 'on/off', 'payment'}
    if any(kw in m.lower() for kw in _PRORATE_KW):
        row['Benchmark Q1 (est*)'] = benchmark_sim[m].iloc[0] * 90 / 365
    comparison_rows.append(row)

comp_df = pd.DataFrame(comparison_rows)
print("=== Overall Simulation Metrics Comparison ===")

# Format for readability
for _, row in comp_df.iterrows():
    m = row['Metric']
    print(f"\n{m}:")
    for col in comp_df.columns[1:]:
        val = row[col]
        if pd.notna(val):
            if 'rate' in m.lower():
                print(f"  {col:>25s}: {val*100:.1f}%")
            elif 'price' in m.lower():
                print(f"  {col:>25s}: ${val:.2f}/MWh")
            elif abs(val) > 1e6:
                print(f"  {col:>25s}: ${val/1e9:.3f}B")
            else:
                print(f"  {col:>25s}: {val:,.1f}")


### 10.2 Cost Decomposition

Compare fixed costs, variable costs, and total costs across scenarios.

In [ ]:
# Load sim outputs for all scenarios
sim_outputs = {}
for scenario_name in all_scenario_names:
    if scenario_name in scenarios:
        rp = scenarios[scenario_name]['results_path']
    else:
        rp = results_benchmark
    sim_outputs[scenario_name] = pd.read_csv(os.path.join(rp, 'overall_simulation_output.csv'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Stacked bar: fixed vs variable costs
ax = axes[0]
labels = all_scenario_names
x = range(len(labels))
fixed = []
variable = []
for sn in labels:
    so = sim_outputs[sn]
    scale = 90 / 365 if sn == 'Benchmark Q1' else 1.0  # pro-rata benchmark
    fixed.append(so['Total fixed costs'].iloc[0] / 1e9 * scale)
    variable.append(so['Total generation costs'].iloc[0] / 1e9 * scale)
bar_colors = [colors_map[sn] for sn in labels]
ax.bar(x, fixed, label='Fixed Costs', color=bar_colors, alpha=0.6, edgecolor='black', linewidth=0.5)
ax.bar(x, variable, bottom=fixed, label='Variable Costs', color=bar_colors, alpha=0.9, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Cost ($B, 90-day equivalent)')
ax.set_title('Cost Decomposition')
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' ', '\n') for s in labels], fontsize=8)
ax.legend()

# (b) Reliability metrics
ax = axes[1]
bar_w = 0.35
load_shed = []
overgen = []
for sn in labels:
    so = sim_outputs[sn]
    scale = 90 / 365 if sn == 'Benchmark Q1' else 1.0
    load_shed.append(so['Total load shedding'].iloc[0] * scale)
    overgen.append(so['Total over generation'].iloc[0] / 1e3 * scale)
ax.bar([v - bar_w/2 for v in x], load_shed, bar_w, label='Load Shedding (MWh)', color='red', alpha=0.7)
ax2 = ax.twinx()
ax2.bar([v + bar_w/2 for v in x], overgen, bar_w, label='Overgeneration (GWh)', color='purple', alpha=0.7)
ax.set_ylabel('Load Shedding (MWh)')
ax2.set_ylabel('Overgeneration (GWh)')
ax.set_title('Reliability Metrics (90-day equivalent)')
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' ', '\n') for s in labels], fontsize=8)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()
plt.close('all')

# Print key differences
for sn in labels:
    so = sim_outputs[sn]
    scale = 90 / 365 if sn == 'Benchmark Q1' else 1.0
    print(f"\n{sn}:")
    print(f"  Total costs (90d eq): ${so['Total costs'].iloc[0] * scale / 1e9:.3f}B")
    print(f"  Avg price: ${so['Cumulative average price'].iloc[0]:.2f}/MWh")
    print(f"  On/offs (90d eq): {so['Total on/offs'].iloc[0] * scale:,.0f}")

### 10.3 LMP Distribution Comparison

Overlaid CDFs and statistics for all scenarios.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) CDF comparison — all scenarios
ax = axes[0]
for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name]
    c = colors_map[scenario_name]
    all_lmps = bus_df[lmp_col_name].dropna().sort_values()
    cdf = np.arange(1, len(all_lmps) + 1) / len(all_lmps)
    ax.plot(all_lmps, cdf, color=c, linewidth=1, label=scenario_name, alpha=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('DA LMP ($/MWh)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('CDF of Bus-Hour DA LMPs')
ax.legend(fontsize=8)
ax.set_xlim(-1100, 500)

# (b) Box plots
ax = axes[1]
data_for_box = []
labels_box = []
box_colors = []
for scenario_name in all_scenario_names:
    bus_df = bus_dfs[scenario_name]
    clipped = bus_df[lmp_col_name].clip(-200, 200)
    data_for_box.append(clipped.values)
    labels_box.append(scenario_name.replace(' ', '\n'))
    box_colors.append(colors_map[scenario_name])

bp = ax.boxplot(data_for_box, labels=labels_box, patch_artist=True,
                whis=[5, 95], showfliers=False)
for patch, c in zip(bp['boxes'], box_colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax.set_ylabel('DA LMP ($/MWh)')
ax.set_title('LMP Distribution (5th-95th percentile)')
ax.axhline(0, color='red', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()
plt.close('all')

### 10.4 Congestion Pattern Differences

Compare which lines are congested under PTDF vs B-theta network modeling.

In [ ]:
# Merge congestion data from all scenarios
cf_ptdf = congestion_freqs['PTDF UC+ED'].set_index('Line')['pct_at_capacity'].rename('PTDF')
cf_uc = congestion_freqs['Btheta UC-only'].set_index('Line')['pct_at_capacity'].rename('Btheta')
cf_merged = pd.concat([cf_ptdf, cf_uc], axis=1).fillna(0)

if _bench_available:
    cf_bench = congestion_freqs['Benchmark Q1'].set_index('Line')['pct_at_capacity'].rename('Benchmark')
    cf_merged = pd.concat([cf_merged, cf_bench], axis=1).fillna(0)

n_plots = 3 if _bench_available else 2
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))

# (a) Scatter: PTDF vs Btheta
ax = axes[0]
ax.scatter(cf_merged['Btheta'], cf_merged['PTDF'], s=10, alpha=0.5, color='purple')
ax.plot([0, 100], [0, 100], 'k--', linewidth=0.5, label='Equal')
ax.set_xlabel('Btheta UC-only: % Hours Congested')
ax.set_ylabel('PTDF UC+ED: % Hours Congested')
ax.set_title('Line Congestion: PTDF vs Btheta')
ax.legend()

# (b) Lines with largest congestion differences
cf_merged['diff_pb'] = cf_merged['PTDF'] - cf_merged['Btheta']
cf_merged['abs_diff_pb'] = cf_merged['diff_pb'].abs()
top_diff = cf_merged.sort_values('abs_diff_pb', ascending=False).head(15)

ax = axes[1]
colors_bar = ['steelblue' if d > 0 else 'darkorange' for d in top_diff['diff_pb']]
ax.barh(range(len(top_diff)), top_diff['diff_pb'], color=colors_bar, alpha=0.8)
ax.set_yticks(range(len(top_diff)))
ax.set_yticklabels([f"Line {int(l)}" for l in top_diff.index], fontsize=8)
ax.set_xlabel('Congestion Difference (PTDF - Btheta, pp)')
ax.set_title('Largest Congestion Differences')
ax.axvline(0, color='black', linewidth=0.5)

# (c) If benchmark available: scatter Benchmark vs Btheta
if _bench_available:
    ax = axes[2]
    ax.scatter(cf_merged['Btheta'], cf_merged['Benchmark'], s=10, alpha=0.5, color='green')
    ax.plot([0, 100], [0, 100], 'k--', linewidth=0.5, label='Equal')
    ax.set_xlabel('Btheta UC-only (90d): % Hours Congested')
    ax.set_ylabel('Benchmark Q1 (365d): % Hours Congested')
    ax.set_title('Line Congestion: Benchmark vs Btheta')
    ax.legend()

plt.tight_layout()
plt.show()
plt.close('all')

# Print key differences
more_congested_ptdf = (cf_merged['PTDF'] > cf_merged['Btheta'] + 5).sum()
more_congested_btheta = (cf_merged['Btheta'] > cf_merged['PTDF'] + 5).sum()
print(f"\nLines >5pp more congested under PTDF: {more_congested_ptdf}")
print(f"Lines >5pp more congested under Btheta: {more_congested_btheta}")
if _bench_available:
    more_congested_bench = (cf_merged['Benchmark'] > cf_merged['Btheta'] + 5).sum()
    more_congested_uc = (cf_merged['Btheta'] > cf_merged['Benchmark'] + 5).sum()
    print(f"Lines >5pp more congested under Benchmark Q1 vs Btheta: {more_congested_bench}")
    print(f"Lines >5pp more congested under Btheta vs Benchmark Q1: {more_congested_uc}")

### 10.5 UC-Only vs UC+ED Impact

The Btheta UC-only scenario removes economic dispatch (SCED), so prices reflect only unit commitment shadow prices. This section quantifies the impact on price volatility and dispatch patterns.

Note: The comparison between PTDF UC+ED and Btheta UC-only conflates two effects (network model + dispatch mode). The benchmark Q1 comparison in Section 10.6 helps disentangle them.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (a) Hourly system LMP comparison
ax = axes[0, 0]
for scenario_name in all_scenario_names:
    c = colors_map[scenario_name]
    sh = sys_hourlys[scenario_name]
    ax.plot(sh['Datetime'], sh['sys_mean_lmp'], color=c, alpha=0.5, linewidth=0.5, label=scenario_name)
ax.set_ylabel('Load-Weighted DA LMP ($/MWh)')
ax.set_title('Hourly System LMP Over Q1')
ax.legend()
ax.axhline(0, color='black', linewidth=0.3)

# (b) LMP volatility (rolling std)
ax = axes[0, 1]
for scenario_name in all_scenario_names:
    c = colors_map[scenario_name]
    sh = sys_hourlys[scenario_name].sort_values('Datetime')
    rolling_std = sh['sys_mean_lmp'].rolling(24).std()
    ax.plot(sh['Datetime'], rolling_std, color=c, alpha=0.6, linewidth=0.6, label=scenario_name)
ax.set_ylabel('24h Rolling Std of LMP ($/MWh)')
ax.set_title('LMP Volatility Comparison')
ax.legend()

# (c) Daily average price comparison (using already-loaded sys_hourlys)
ax = axes[1, 0]
for scenario_name in all_scenario_names:
    c = colors_map[scenario_name]
    sh = sys_hourlys[scenario_name].copy().sort_values('Datetime')
    sh['_date'] = sh['Datetime'].dt.date
    sh['_wt'] = sh['sys_mean_lmp'] * sh['sys_demand']
    daily = sh.groupby('_date').agg(_wt_sum=('_wt', 'sum'), _d_sum=('sys_demand', 'sum'))
    daily_lmp = daily['_wt_sum'] / daily['_d_sum']
    ax.plot(range(len(daily_lmp)), daily_lmp.values, color=c, alpha=0.7, linewidth=1,
            label=scenario_name, linestyle='--' if 'Benchmark' in scenario_name else '-')
ax.set_xlabel('Day of Q1')
ax.set_ylabel('Daily Load-Weighted LMP ($/MWh)')
ax.set_title('Daily Average System LMP')
ax.legend()

# (d) Commitment differences (on/offs)
ax = axes[1, 1]
labels = all_scenario_names
on_offs = [sim_outputs[sn]['Total on/offs'].iloc[0] for sn in labels]
bar_colors = [colors_map[sn] for sn in labels]
ax.bar(range(len(labels)), on_offs, color=bar_colors, alpha=0.8)
ax.set_ylabel('Total On/Off Transitions')
ax.set_title('Generator Commitment Transitions')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9, rotation=15)

plt.tight_layout()
plt.show()
plt.close('all')

# Print summary
for sn in all_scenario_names:
    so = sim_outputs[sn]
    print(f"\n{sn}:")
    print(f"  Avg price: ${so['Cumulative average price'].iloc[0]:.2f}/MWh")
    print(f"  Total on/offs: {so['Total on/offs'].iloc[0]:,}")
    print(f"  Load shedding: {so['Total load shedding'].iloc[0]:.1f} MWh")
    print(f"  Reserve shortfall: {so['Total reserve shortfall'].iloc[0]:.1f} MWh")

### 10.6 Benchmark Comparison: 90-Day Results vs 365-Day Q1

Extract Q1 (Jan–Mar) from the 365-day benchmark (btheta UC+ED) and compare with both 90-day scenarios. This partially disentangles network model and dispatch mode effects:
- **Benchmark Q1 vs Btheta UC-only**: Same network (btheta), different dispatch → isolates ED effect
- **Benchmark Q1 vs PTDF UC+ED**: Same dispatch (UC+ED), different network → isolates PTDF effect

In [ ]:
# Benchmark Q1 was already loaded in cell 19 — use it directly
# Daily demand-weighted LMP comparison (all 3 scenarios)
print("=== Q1 Load-Weighted System LMP ===")
for scenario_name in all_scenario_names:
    sh = sys_hourlys[scenario_name]
    total_d = sh['sys_demand'].sum()
    if total_d > 0:
        wt_lmp = (sh['sys_mean_lmp'] * sh['sys_demand']).sum() / total_d
    else:
        wt_lmp = 0
    print(f"  {scenario_name}: ${wt_lmp:.2f}/MWh")

fig, ax = plt.subplots(figsize=(16, 5))
for scenario_name in all_scenario_names:
    sh = sys_hourlys[scenario_name].copy().sort_values('Datetime')
    sh['_date'] = sh['Datetime'].dt.date
    sh['_wt'] = sh['sys_mean_lmp'] * sh['sys_demand']
    daily = sh.groupby('_date').agg(_wt_sum=('_wt', 'sum'), _d_sum=('sys_demand', 'sum'))
    daily_lmp = daily['_wt_sum'] / daily['_d_sum']
    c = colors_map[scenario_name]
    ax.plot(range(len(daily_lmp)), daily_lmp.values, color=c, linewidth=1.5,
            label=scenario_name, alpha=0.8,
            linestyle='--' if 'Benchmark' in scenario_name else '-')

ax.axhline(0, color='black', linewidth=0.3)
ax.set_xlabel('Day of Q1')
ax.set_ylabel('Daily Load-Weighted System LMP ($/MWh)')
ax.set_title('Q1 Daily System LMP: 3-Way Comparison')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
plt.close('all')

# Negative LMP comparison
print("\n=== Q1 Negative LMP Prevalence ===")
for scenario_name in all_scenario_names:
    bdf = bus_dfs[scenario_name]
    neg_pct = (bdf[lmp_col_name] < 0).mean() * 100
    print(f"  {scenario_name}: {neg_pct:.1f}% bus-hours negative")

# Overgeneration comparison
print("\n=== Q1 Total Overgeneration ===")
for scenario_name in all_scenario_names:
    sh = sys_hourlys[scenario_name]
    print(f"  {scenario_name}: {sh['sys_overgen'].sum():,.0f} MWh")

## 11. Key Findings & Observations

Summary of insights from the 3-way scenario comparison.

In [ ]:
print("=" * 80)
print("KEY FINDINGS — 90-DAY COMPARATIVE PCM ANALYSIS")
print("=" * 80)

# Overall cost comparison
ptdf_total = sim_outputs['PTDF UC+ED']['Total costs'].iloc[0]
uc_total = sim_outputs['Btheta UC-only']['Total costs'].iloc[0]
print(f"\n1. TOTAL COSTS (90 days):")
print(f"   PTDF UC+ED:     ${ptdf_total/1e9:.3f}B")
print(f"   Btheta UC-only: ${uc_total/1e9:.3f}B")
print(f"   Difference:     ${(ptdf_total - uc_total)/1e6:.1f}M ({(ptdf_total/uc_total - 1)*100:+.1f}%)")

# Price comparison
ptdf_price = sim_outputs['PTDF UC+ED']['Cumulative average price'].iloc[0]
uc_price = sim_outputs['Btheta UC-only']['Cumulative average price'].iloc[0]
print(f"\n2. AVERAGE PRICE:")
print(f"   PTDF UC+ED:     ${ptdf_price:.2f}/MWh")
print(f"   Btheta UC-only: ${uc_price:.2f}/MWh")
print(f"   Difference:     ${ptdf_price - uc_price:+.2f}/MWh")

# Commitment
ptdf_onoff = sim_outputs['PTDF UC+ED']['Total on/offs'].iloc[0]
uc_onoff = sim_outputs['Btheta UC-only']['Total on/offs'].iloc[0]
print(f"\n3. UNIT COMMITMENT TRANSITIONS:")
print(f"   PTDF UC+ED:     {ptdf_onoff:,}")
print(f"   Btheta UC-only: {uc_onoff:,}")
print(f"   Difference:     {ptdf_onoff - uc_onoff:+,}")

# Reliability
print(f"\n4. RELIABILITY:")
print(f"   Load shedding — PTDF: {sim_outputs['PTDF UC+ED']['Total load shedding'].iloc[0]:.1f} MWh, "
      f"UC-only: {sim_outputs['Btheta UC-only']['Total load shedding'].iloc[0]:.1f} MWh")
print(f"   Reserve shortfall — PTDF: {sim_outputs['PTDF UC+ED']['Total reserve shortfall'].iloc[0]:.1f} MWh, "
      f"UC-only: {sim_outputs['Btheta UC-only']['Total reserve shortfall'].iloc[0]:.1f} MWh")

# Congestion
more_ptdf = (cf_merged['PTDF'] > cf_merged['Btheta'] + 5).sum()
more_btheta = (cf_merged['Btheta'] > cf_merged['PTDF'] + 5).sum()
print(f"\n5. CONGESTION DIFFERENCES:")
print(f"   Lines >5pp more congested under PTDF:  {more_ptdf}")
print(f"   Lines >5pp more congested under Btheta: {more_btheta}")

# Negative LMP
print(f"\n6. NEGATIVE LMP PREVALENCE:")
for scenario_name in scenarios:
    bdf = bus_dfs[scenario_name]
    neg_pct = (bdf[lmp_col_name] < 0).mean() * 100
    floor_pct = (bdf[lmp_col_name] == -1000).mean() * 100
    print(f"   {scenario_name}: {neg_pct:.1f}% negative, {floor_pct:.1f}% at floor")

print(f"\n7. RENEWABLES:")
print(f"   Both scenarios: 0 MWh curtailment, ~{sim_outputs['PTDF UC+ED']['Overall renewables penetration rate'].iloc[0]*100:.1f}% penetration")

print("\n" + "=" * 80)
print("INTERPRETATION:")
print("=" * 80)
print("""
The PTDF network model with UC+ED results in slightly higher total costs than
the Btheta UC-only configuration. This reflects two confounded effects:
  (a) PTDF more accurately models power flow distribution, potentially
      identifying congestion that B-theta misses
  (b) Economic dispatch (SCED) redispatches units based on real-time conditions

The benchmark Q1 comparison helps disentangle these effects. Key questions
for further analysis:
  - Does PTDF congestion align better with AC power flow solutions?
  - Does the cost increase from PTDF reflect real system costs or model artifacts?
  - How does removing ED affect system reliability in longer simulations?
""")